# Spending Assistant notebook (Gemini-ready)

This updated notebook cleans the transactions file, runs quick EDA, and adds a Gemini API helper you can test instead of Ollama.

## Before you run
- Install once: `pip install google-generativeai pandas matplotlib seaborn`
- Set your key as an environment variable named `GEMINI_API_KEY` (never hardcode it in the notebook): `GEMINI_API_KEY = "your-key"`
- Or use an environment variable named `GEMINI_API_KEY`


In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_CANDIDATES = [
    "sample_transactions.csv",
]

def find_data_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"None of these files were found: {candidates}")

DATA_FILE = find_data_file(DATA_CANDIDATES)
print(f"Using data file: {DATA_FILE}")


In [ ]:
df_raw = pd.read_csv(DATA_FILE)
print("Exploratory Data Analysis")
print("=" * 60)
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")
print(f"Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("Data types:")
print(df_raw.dtypes)

missing_count = df_raw.isnull().sum()
missing_pct = (missing_count / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct}).sort_values("missing_pct", ascending=False)
print("Missing values report:")
print(missing_df[missing_df["missing_count"] > 0] if (missing_df["missing_count"] > 0).any() else "No missing values found")

print(f"Duplicate rows: {df_raw.duplicated().sum()}")
display(df_raw.head())
df_clean = df_raw.copy()
df_clean.columns = [df_clean.columns[i].strip().lower().replace(" ", "_") for i in range(len(df_clean.columns))]

In [ ]:
print(df_raw.columns.tolist())
print(df_raw.head(3))

In [ ]:
df_clean = df_raw.copy()
df_clean.columns = [c.strip().lower().replace(" ", "_") for c in df_clean.columns]

print("Normalized columns:", list(df_clean.columns))

rename_map = {
    "transaction_date": "transaction_date",
    "description": "merchant",
    "spend_category": "category",
    "amount_($)": "amount",
}

df_clean = df_clean.rename(columns=rename_map)

required_cols = ["transaction_date", "merchant", "category", "amount"]
missing_required = [c for c in required_cols if c not in df_clean.columns]
if missing_required:
    raise ValueError(f"Still missing columns: {missing_required}")

df_clean["transaction_date"] = pd.to_datetime(df_clean["transaction_date"], format="%d-%b", errors="coerce")
df_clean["amount"] = pd.to_numeric(df_clean["amount"], errors="coerce")
df_clean["merchant"] = df_clean["merchant"].astype(str).str.strip().str.title()
df_clean["category"] = df_clean["category"].astype(str).str.strip().str.title()

df_clean = df_clean.dropna(subset=["transaction_date", "merchant", "category", "amount"]).copy()
df_clean = df_clean.sort_values("transaction_date").reset_index(drop=True)
df_clean["month"] = df_clean["transaction_date"].dt.to_period("M").astype(str)

display(df_clean.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_clean["amount"].hist(bins=30, color="#2a9d8f", alpha=0.85, ax=axes[0])
axes[0].set_title("Distribution of transaction amounts")
axes[0].set_xlabel("Amount")
axes[0].set_ylabel("Count")

monthly_spend = df_clean.groupby("month", as_index=False)["amount"].sum()
sns.lineplot(data=monthly_spend, x="month", y="amount", marker="o", ax=axes[1], color="#264653")
axes[1].set_title("Monthly spend trend")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Total amount")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print("Top categories by spend:")
display(df_clean.groupby("category", as_index=False)["amount"].sum().sort_values("amount", ascending=False).head(10))


In [ ]:
import os
import time
import pandas as pd

try:
    from google import genai
    from google.genai import types
except ImportError:
    raise ImportError("Install first with: pip install google-genai")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    print("Set GEMINI_API_KEY in your environment before running the Gemini cells.")

MODEL_NAME = "gemini-2.5-flash"


def build_spending_context(question: str, df: pd.DataFrame) -> str:
    q = question.lower().strip()
    total_spend = df["amount"].sum()
    monthly = df.groupby("month")["amount"].sum().sort_index()
    top_categories = df.groupby("category")["amount"].sum().sort_values(ascending=False).head(8)
    top_merchants = df.groupby("merchant")["amount"].sum().sort_values(ascending=False).head(8)

    parts = [
        f"Question: {question}",
        f"Total spend: ${total_spend:,.2f}",
        "Monthly totals:",
        monthly.to_string(),
        "Top categories:",
        top_categories.to_string(),
        "Top merchants:",
        top_merchants.to_string(),
    ]

    if "nov" in q or "november" in q:
        nov = df[df["transaction_date"].dt.month == 11]["amount"].sum()
        parts.append(f"November total: ${nov:,.2f}")
    if "dec" in q or "december" in q:
        dec = df[df["transaction_date"].dt.month == 12]["amount"].sum()
        parts.append(f"December total: ${dec:,.2f}")

    return "\n\n".join(parts)


def ask_gemini(question: str, df: pd.DataFrame, model_name: str = MODEL_NAME) -> str:
    if not GEMINI_API_KEY:
        raise ValueError("Missing Gemini API key. Set GEMINI_API_KEY first.")

    client = genai.Client(api_key=GEMINI_API_KEY)
    prompt = f"""You are a concise personal finance assistant.
Use only the supplied spending context.
If the answer is uncertain, say so.
Respond in 2 to 4 sentences.

{build_spending_context(question, df)}
"""

    delays = [0, 2, 4]
    last_error = None
    for delay in delays:
        if delay:
            time.sleep(delay)
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.2),
            )
            return (getattr(response, "text", "") or "No response returned by Gemini.").strip()
        except Exception as e:
            last_error = e
            if "503" not in str(e) and "UNAVAILABLE" not in str(e):
                raise
    raise RuntimeError("Gemini is busy right now. Please try again in a few moments.") from last_error


In [ ]:
question = "What are my top spending categories?"
# question = "What is my total spending in November?"
# question = "Which merchant did I spend the most on?"

try:
    answer = ask_gemini(question, df_clean)
    print(answer)
except Exception as e:
    print(f"Gemini test could not run yet: {e}")


In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

print("--- Models Available to My API Key ---")
for model in client.models.list():
    if "generateContent" in model.supported_actions:
        print(model.name)